In [217]:
from geometry_msgs.msg import Pose
import numpy as np

accepted_codes = ("G0", "G1", "G90", "G91")
rejected_actuations = ("E", "F")

def clean_gcode_line(l):
    split_actuations = l.split(" ")
    cleaned_actuations = list(filter(lambda l: not l.startswith(rejected_actuations), split_actuations))
    joined_actuations = " ".join(cleaned_actuations)
    return joined_actuations

def generate_waypoints(gcode_lines: list[str]):
    # Holds the presently parsed waypoint
    coords = {"x": 0.0, "y": 0.0, "z": 0.0}
    waypoints = []

    for line in gcode_lines:
        for code in line.split(" "):
            match code[0]:
                case "X":
                    coords["x"] = float(code[1:])
                case "Y":
                    coords["y"] = float(code[1:])
                case "Z":
                    coords["z"] = float(code[1:])

        waypoints.append(np.array([
            coords["x"],
            coords["y"],
            coords["z"]
        ]))

    waypoints = np.array(waypoints)

    return waypoints    

In [230]:
import plotly.express as px

def display_waypoints(waypoints):
    x = waypoints[:, 0]
    y = waypoints[:, 1]
    z = waypoints[:, 2]
    s = np.arange(len(waypoints))

    fig = px.scatter_3d(
        x=x,
        y=y,
        z=z,
        color=s,
        color_continuous_scale="Magma",
        labels={
            "x": "X Axis (mm)",
            "y": "Y Axis (mm)", 
            "z": "Z Axis (mm)",
            "color": "Sequential order"},
        title="Interactive Plotly 3D Scatter Plot with 4D Intensity",
        opacity=0.8,
    )


    fig.update_traces(marker=dict(size=5))

    fig.show()

In [231]:
print_name = "box"

with open(f"../gcode/{print_name}.gcode", "r") as file:
    gcode_lines = file.readlines()

filtered_lines = list(filter(lambda l: l.startswith(accepted_codes), gcode_lines))
parsed_lines = list(map(lambda l: clean_gcode_line(l), filtered_lines))
waypoints = generate_waypoints(parsed_lines)[14:-2]

In [ ]:
distance = np.sqrt(waypoints[:, 0]**2 + waypoints[:, 1]**2)

In [239]:
waypoints = waypoints - waypoints[int(np.argmin(distance))]

In [240]:
display_waypoints(waypoints)

In [241]:
import yaml
import numpy as np

def read_pose_coordinates(pose_filename):
    with open(f"../poses/{pose_filename}.yaml", "r") as file:
        yaml_data = yaml.safe_load(file)

    coords = yaml_data["position"]
    position = np.array([coords['x'], coords['y'], coords['z']])
    return position

In [248]:
p1 = read_pose_coordinates("plane_p1")
p2 = read_pose_coordinates("plane_p2")
p3 = read_pose_coordinates("plane_p3")

u = p2 - p1
v = p3 - p1

n = np.cross(u, v)
a, b, c = n

d = np.dot(n, p1)

print(f"Plane equation: ({a})x + ({b})y + ({c})z = {d}")

Plane equation: (0.00019488394560094403)x + (0.00017991919553185293)y + (0.0024637924767927245)z = 0.0003073449940063747


In [243]:
x_p = u / np.linalg.norm(u)
z_p = n / np.linalg.norm(n)
y_p = np.cross(x_p, z_p) / np.linalg.norm(np.cross(x_p, z_p))

In [249]:
R = np.column_stack((x_p, y_p, z_p))
t = p1

In [264]:
transformed_points = np.apply_along_axis(lambda v: np.dot(R, v), axis=1, arr=waypoints)

In [265]:
display_waypoints(transformed_points)